# 05. Exponential time complexity

지금까지 언급한 일반적인 다항 시간 복잡도 (Polynomial time complexity)와 달리 지수 시간 복잡도 (Exponential time complexity)는 용어 그대로 기하 급수적으로 데이터 처리 시간이 늘어납니다. 1970대부터 다항 시간 복잡도를 가지는 알고리즘을 `P 영역 내에 있는 알고리즘`이라고 명명하였으며, P 영역 내/외를 기준으로 효율적인 알고리즘인지 아닌지를 구분합니다.

|$n$	|$n^2$	|$2^n$|
|---|---|---|
|2|4|4|
|4|16|16|
|8|64|256|
|16|256|65536|

## 시간 복잡도를 P 영역으로 전환하는 법

이번엔 P 영역 밖에 있는 알고리즘을 개선해 P 영역 내로 전환해봅시다. 피보나치 수열이 가장 대표적 예시 중 하나 입니다. 피보나치 수열은 아래와 같은 형태를 지니죠.

$$
0,1,1,2,3,5,8,13,21,34,...
$$

알고리즘의 효율성을 차치하고, 일반적으로 피보나치 수열의 `n + 1 번째 항`을 계산하는 방법은 `n 번째 항`과 `n - 1 번째 항`을  더해서 계산합니다. 이를 코드로 구현하면 다음과 같죠.

In [6]:
def fib(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fib(n - 1) + fib(n - 2)

fib(40)

102334155

이 코드를 사용해서 피보나치 수열의 10 번째 항을 구하려면 몇 번의 함수 호출이 필요할까요?
| n    | 함수 호출 횟수 |
|------|----------------|
| 0    | 1              |
| 1    | 1              |
| 2    | 3              |
| 3    | 5              |
| 4    | 9              |
| 5    | 15             |
| 6    | 25             |
| 7    | 41             |
| 8    | 67             |
| 9    | 109            |
| 10   | 177            |

대략 177 회의 함수 호출이 필요합니다. 심지어 이 계산 과정은 재귀적으로 호출되기 때문에 최초 fib(10)은 메모리에서 사라지지 않고 계속 남아 있어요. 이 함수의 시간 복잡도는 $O(n^2)$입니다. 그럼 이런 함수를 어떻게 개선할까요?

## 예제 22. 피보나치 수열의 계산 알고리즘 개선

피보나치 수열 계산 알고리즘을 개선하기 위한 의사 코드를 봅시다.

- n + 1 번째 피보나치 수를 알고 싶습니다. (index를 기준으로 하므로 n ≥ 0 입니다.)
    - 만약 n ≤ 1 이면, n을 그대로 반환합니다.
    - `조부모 수 = 0`, `부모 수 = 1`로 지정하고, `자식 수`를 각 회차의 피보나치 수로 가정합시다.
    - 아래 반복문을 n - 1 회 진행합니다.
        - `자식 수 = 조부모 수 + 부모 수` 입니다.
        - 부모 수를 조부모 수에 다시 할당합니다. (`조부모 수 = 부모 수`)
        - 자식 수를 부모 수에 다시 할당합니다. (`부모 수 = 자식 수`)
    - 반복문이 끝나면 `자식 수`를 반환합니다.

In [7]:
def fib(n):
    if n <= 1:
        return n
    
    grandparent = 0
    parent = 1
    for i in range(n - 1):
        current = grandparent + parent
        grandparent = parent
        parent = current
    return current

fib(1000)

43466557686937456435688527675040625802564660517371780402481729089536555417949051890403879840079255169295922593080322634775209689623239873322471161642996440906533187938298969649928516003704476137795166849228875

위와 같이 만들어진 계산 알고리즘은 $O(n)$의 시간 복잡도를 가집니다. 당장 기존 코드에서 피보나치의 40 번째 수를 찾는 데만 해도 10초 정도의 시간이 소요되는데, 개선된 코드는 1000 번째 수를 찾는 데도 거의 즉시 결과를 반환합니다. 엄청난 발전이네요.

## 번외 예제 22. 피보나치 알고리즘을 `더` 개선하기

- 다들 생소하실텐데, 피보나치 수열에는 다음과 같은 성질이 있습니다. ($F(n)$을 피보나치의 n 번째 수라고 합시다.)

$$
F(2k+1) = F(k)\{2F(k+1)-F(k)\} \\
F(2k) = {F(k)}^2 + {F(k+1)}^2 \\
\text{(k} \ge \text{1 일 때)}
$$

이 수식을 활용하면 더 빠른 알고리즘을 구현할 수 있습니다.

In [ ]:
def fib_fast(n: int) -> int:
    def helper(k: int):
        # returns (F(k), F(k+1))
        if k == 0:
            return (0, 1)
        a, b = helper(k >> 1)          # a=F(m), b=F(m+1), m=k//2
        c = a * ((b << 1) - a)         # F(2m)
        d = a * a + b * b              # F(2m+1)
        if k & 1:
            return (d, c + d)          # (F(2m+1), F(2m+2))
        else:
            return (c, d)              # (F(2m), F(2m+1))

    return helper(n)[0]

# fib vs fib_fast
import time
fib_num = 1_000_000
start_time = time.time()
fib(fib_num)
print("fib took", time.time() - start_time, "seconds")

start_time = time.time()
fib_fast(fib_num)
print("fib_fast took", time.time() - start_time, "seconds")

fib took 4.6636717319488525 seconds
fib_fast took 0.03439640998840332 seconds


## $O(k^n)$

$O(k^n)$의 복잡도를 가진 알고리즘은 가장 대표적인 지수 시간 복잡도를 가진 알고리즘입니다. 이런 시간 복잡도를 가진 알고리즘은 데이터가 조금만 커져도 시간이 너무 오래 걸리기 때문에 큰 쓸 모가 없습니다. 예전에 번호키가 달린 핸드폰에 기입된 천지인이나 영어 키패드를 떠올려 보시면, 어떤 특정 글자를 만들기 위해 번호를 어떠한 순서로 눌러야 합니다. 근데 반대로, 어떤 숫자를 주고 **이걸로 만들 수 있는 글자가 어떤 것들이 있냐**라면 어떨까요?

<img src="https://img.notionusercontent.com/s3/prod-files-secure%2Fbcdd7a84-a71a-4d71-9efe-900d61cae82f%2Fea4285a1-4a95-445c-8219-1973b0a2cad6%2Fimage.png/size/w=1500?exp=1772683000&sig=shjizNYjAQRNuNPlm7Rv8hzD9qwtlrjWG3SXR8A02yk&id=2ffe2cae-64bd-8022-ade8-da40c3335a85&table=block&userId=a9e624ed-17cf-4575-903f-387405ea2d65">

예를 들어, 위 키패드에서 234 라는 숫자는 [a,b,c][d,e,f][g,h,i] 순서의 모든 조합, 즉 $3^3$ 개의 조합이 나옵니다. 이 과정은 모든 조합을 탐색해야 합니다. 빠르게 계산할 수 있는 방법이 없네요. 이번엔 이걸 알고리즘으로 한 번 구현해봅시다.

## 예제 23. 문자열 조합 알고리즘

의사 코드 입니다.

- 만약 주어진 글자가 없다면 빈 리스트를 반환합니다.
- 문자열을 반환하기 위한 `결과(result)` 리스트를 생성합니다. 처음에는 빈 문자열 하나만 가지고 있도록 합니다.
- 입력된 숫자 마다 아래의 반복을 수행합니다.
    - 우선, 입력된 숫자가 글자를 나타내는 숫자가 아닐 경우 (예를 들어, 위 키패드를 보면, 1,0 은 문자열을 반환하지 못합니다.) 숫자 에러(`raise ValueError(f”invalid digit: {digit}”`)를 반환합니다.
    - 미리 입력된 `digit_to_letters`의 데이터를 기준으로 해당 숫자에 대응하는 문자를 받아옵니다.
    - 한 번 더 빈 리스트 `new_result`를 만듭니다.
    - 이중 for 문이 필요합니다. `result` 내에 있는 `combo` 마다 / `letters` 내에 있는 `letter` 마다 다음 반복문을 수행합니다.
        - `new_result`에 이미 존재하는 `combo`에 `letter`를 더해서 확장합니다.
    - `result`를 생성된 `new_result`로 대체합니다.
- 최종적으로 `result`를 반환합니다.

In [24]:
def letter_combinations(digits):
    if not digits:
        return []
    
    result = [""]
    for digit in digits:
        if digit not in digit_to_letters:
            raise ValueError(f"Invalid digit: {digit}")
        letters = digit_to_letters[digit]

        new_result = []

        for combo in result:
            for letter in letters:
                new_result.append(combo + letter)

        result = new_result

    return result

# Don't touch below this line

digit_to_letters = {
    "2": "abc",
    "3": "def",
    "4": "ghi",
    "5": "jkl",
    "6": "mno",
    "7": "pqrs",
    "8": "tuv",
    "9": "wxyz",
}

len(letter_combinations("999"))

64

만약 실제 전화 번호, (앞 010은 제외하고) 4 자리 - 4 자리 구조를 사용한다면, 가능한 조합은 작게는 $3^8$ 부터 많게는 $4^8 = 2^{16}=65,536$
 대략 6만 5천 개 즈음 나오겠네요. 1초에 100 만 개의 조합을 처리한다 쳐도, 단순히 가능한 문자열 조합을 확인하는 데만, 0.065초가 걸립니다.

## 지수적 증가 수열

어떤 SNS 내 인플루언서의 팔로워 수가 지수적으로 증가한다고 할 때, n 일 후에 몇 명이나 될지 알아봅시다.

## 예제 24. 지수적 증가 수열

- 현재 팔로워 수 $n$, 증분 상수 $factor$, 경과일 $day$ 를 인수로 받아, 수열을 순차적으로 담은 리스트를 반환하는 함수를 작성해 봅시다.

- $O(n\ log(n))$ 으로 작성

In [27]:
"""
이 경우 계산을 n 번 반복하므로 O(n) 이고, 
그 과정에서 거듭 제곱도 n 번 하여 O(log n) 이기 때문에
최종적으로 시간 복잡도는 O(n log n)이 됩니다.
"""
def exponential_growth_nlogn(n, factor, days):
    result = []
    for i in range(days + 1):
        result.append(n * (factor ** i))
    return result
    
# 위 식을 간단하게 한 줄로 바꾸면
def exponential_growth_nlogn(n, factor, days):
    return [n * (factor ** i) for i in range(days + 1)]

- $O(n)$ 으로 작성

In [28]:
"""
이 경우 계산을 n 번 반복하므로 O(n) 이고,
그 과정에서 단순 곱셈만 하므로 O(1) 이기 때문에
최종적으로 시간 복잡도는 O(n)이 됩니다.
"""
def exponential_growth_n(n, factor, days):
    result = []
    for i in range(days + 1):
        result.append(n)
        n *= factor
    return result